# Imports

In [164]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict
import os
import csv
import json

print("✓ Librerías importadas")

✓ Librerías importadas


# Configuración y definición de parámetros

In [165]:
VIDEO_PATH = "VideosAnalisis\\clip 2 ‐ Hecho con Clipchamp.mp4"
MAPA_PATH = "beachvolleyballcourt.png"
MODEL_PATH = "yolo11n.pt"

OUTPUT_TRACKING_DIR = "outputs/player_tracking"
OUTPUT_FIELD_DIR = "outputs/field"
MAP_POINTS_JSON = "map_points.json"

EXPECTED_PLAYERS = 4
DETECTION_ZONE_EXPAND_X = 0.15
DETECTION_ZONE_EXPAND_Y = 0.25

TRACKER_MAX_AGE = 15
TRACKER_MIN_HITS = 3
TRACKER_IOU_THRESHOLD = 0.5
TRACKER_COLOR_WEIGHT = 0.2
TRACKER_POSITION_WEIGHT = 0.8
TRACKER_MIN_SIMILARITY = 0.3
TRACKER_MAX_MOVEMENT = 0.05

AUTO_OPEN_VIDEO = True
GENERATE_INDIVIDUAL_TRAJECTORIES = True

os.makedirs(OUTPUT_TRACKING_DIR, exist_ok=True)
os.makedirs(OUTPUT_FIELD_DIR, exist_ok=True)

model = YOLO(MODEL_PATH)


# Definición de funciones Auxiliares

In [166]:
def get_points(event, x, y, flags, params):
    points = params["points"]
    image = params["image"]
    wname = params["wname"]
    max_points = params["max_points"]
    if event == cv2.EVENT_LBUTTONDOWN and len(points) < max_points:
        points.append([x, y])
        cv2.circle(image, (x, y), 6, (0, 0, 255), -1)
        cv2.putText(image, str(len(points)), (x + 5, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)
        cv2.imshow(wname, image)
        if len(points) == max_points:
            cv2.waitKey(300)
            cv2.destroyWindow(wname)

def expand_field_zone(field_points, expand_x=0.15, expand_y=0.20):
    center = np.mean(field_points, axis=0)
    x_coords = field_points[:, 0]
    y_coords = field_points[:, 1]
    field_width = np.max(x_coords) - np.min(x_coords)
    field_height = np.max(y_coords) - np.min(y_coords)
    
    expanded_points = []
    for pt in field_points:
        direction = pt - center
        if abs(direction[0]) > 1e-6:
            expand_x_pixels = field_width * expand_x / 2
            direction[0] += np.sign(direction[0]) * expand_x_pixels
        if abs(direction[1]) > 1e-6:
            expand_y_pixels = field_height * expand_y / 2
            direction[1] += np.sign(direction[1]) * expand_y_pixels
        expanded_points.append(center + direction)
    
    return np.array(expanded_points, dtype=np.float32)

def get_player_color(track_id, all_ids, position_x=None, field_center_x=None):
    left_team_colors = [(255, 100, 0), (180, 0, 0)]
    right_team_colors = [(0, 100, 255), (0, 0, 180)]
    
    if position_x is not None and field_center_x is not None:
        sorted_ids = sorted(all_ids)
        if track_id in sorted_ids:
            team_player_idx = sorted_ids.index(track_id) % 2
            if position_x < field_center_x:
                return left_team_colors[team_player_idx]
            else:
                return right_team_colors[team_player_idx]
    
    return (200, 200, 200)

# Definición del PlayerTracker

In [175]:
from scipy.optimize import linear_sum_assignment

class PlayerTracker:
    def __init__(self, max_age=30, min_hits=3, iou_threshold=0.3, 
                 color_weight=0.2, position_weight=0.8, max_players=4,
                 max_movement_percent=0.08, field_center_x=None):
        self.max_age = max_age
        self.min_hits = min_hits
        self.color_weight = color_weight
        self.position_weight = position_weight
        self.max_players = max_players
        self.max_movement = max_movement_percent
        self.field_center_x = field_center_x
        self.tracks = {}
        self.next_id = 1
        self.frame_count = 0
        self.diagonal = None
    
    def _init_frame(self, frame):
        if self.diagonal is None:
            h, w = frame.shape[:2]
            self.diagonal = np.sqrt(w**2 + h**2)
    
    def _extract_histogram(self, frame, bbox):
        x1, y1, x2, y2 = [max(0, int(v)) for v in bbox]
        x2, y2 = min(frame.shape[1], x2), min(frame.shape[0], y2)
        if x2 <= x1 or y2 <= y1:
            return None
        
        roi = frame[y1:y2, x1:x2]
        h, w = roi.shape[:2]
        mx, my = int(w * 0.15), int(h * 0.1)
        if mx > 0 and my > 0 and h > my*2 and w > mx*2:
            roi = roi[my:h-my, mx:w-mx]
        if roi.size == 0:
            return None
        
        hsv = cv2.cvtColor(roi, cv2.COLOR_BGR2HSV)
        hist_h = cv2.calcHist([hsv], [0], None, [50], [0, 180])
        hist_s = cv2.calcHist([hsv], [1], None, [60], [0, 256])
        cv2.normalize(hist_h, hist_h, 0, 1, cv2.NORM_MINMAX)
        cv2.normalize(hist_s, hist_s, 0, 1, cv2.NORM_MINMAX)
        return np.concatenate([hist_h.flatten(), hist_s.flatten()])
    
    def _compare_histograms(self, h1, h2):
        if h1 is None or h2 is None:
            return 0.5
        corr_h = cv2.compareHist(h1[:50].reshape(-1,1).astype(np.float32), 
                                  h2[:50].reshape(-1,1).astype(np.float32), cv2.HISTCMP_CORREL)
        corr_s = cv2.compareHist(h1[50:].reshape(-1,1).astype(np.float32), 
                                  h2[50:].reshape(-1,1).astype(np.float32), cv2.HISTCMP_CORREL)
        return (0.6 * corr_h + 0.4 * corr_s + 1) / 2
    
    def _predict_position(self, track):
        positions = track['positions']
        if len(positions) < 2:
            return positions[-1]
        
        recent = positions[-min(4, len(positions)):]
        velocities = [(recent[i][0] - recent[i-1][0], recent[i][1] - recent[i-1][1]) 
                      for i in range(1, len(recent))]
        weights = list(range(1, len(velocities) + 1))
        total_w = sum(weights)
        
        avg_v = (sum(v[0]*w for v,w in zip(velocities, weights)) / total_w,
                 sum(v[1]*w for v,w in zip(velocities, weights)) / total_w)
        
        frames_missed = max(1, track['age'])
        max_move = self.diagonal * self.max_movement * frames_missed
        
        return (positions[-1][0] + np.clip(avg_v[0] * frames_missed, -max_move, max_move),
                positions[-1][1] + np.clip(avg_v[1] * frames_missed, -max_move, max_move))
    
    def _calc_similarity(self, track, det_feat, pred_pos):
        det_pos = det_feat['position']
        last_pos = track['positions'][-1]
        frames_missed = max(1, track['age'] + 1)
        max_dist = self.diagonal * self.max_movement * frames_missed
        
        dist_pred = np.sqrt((det_pos[0] - pred_pos[0])**2 + (det_pos[1] - pred_pos[1])**2)
        dist_last = np.sqrt((det_pos[0] - last_pos[0])**2 + (det_pos[1] - last_pos[1])**2)
        
        # LÍMITE ABSOLUTO: Rechazar distancias imposibles (>300px o >3x max_dist)
        absolute_max = min(300, max_dist * 3)
        if dist_last > absolute_max:
            return 0, 0, 0
        
        # Penalización más agresiva para distancias grandes
        if dist_last > max_dist * 1.5:
            return 0, 0, 0
        
        pos_score = max(0, 1 - dist_pred / max_dist) * 0.7 + max(0, 1 - dist_last / max_dist) * 0.3
        if len(track['positions']) < 3:
            pos_score = max(0, 1 - dist_pred / max_dist) * 0.3 + max(0, 1 - dist_last / max_dist) * 0.7
        
        color_score = self._compare_histograms(track['histogram'], det_feat['histogram'])
        total = self.position_weight * pos_score + self.color_weight * color_score
        return total, pos_score, color_score
    
    def _create_track(self, det_feat):
        self.tracks[self.next_id] = {
            'id': self.next_id, 'bbox': det_feat['bbox'], 'position': det_feat['position'],
            'positions': [det_feat['position']], 'histogram': det_feat['histogram'],
            'age': 0, 'hits': 1, 'matched': True
        }
        self.next_id += 1
    
    def _update_track(self, track, det_feat):
        track['bbox'] = det_feat['bbox']
        track['position'] = det_feat['position']
        track['positions'].append(det_feat['position'])
        if track['histogram'] is not None and det_feat['histogram'] is not None:
            track['histogram'] = 0.2 * det_feat['histogram'] + 0.8 * track['histogram']
        elif det_feat['histogram'] is not None:
            track['histogram'] = det_feat['histogram']
        track['age'] = 0
        track['hits'] += 1
        track['matched'] = True
    
    def _count_players_per_side(self):
        """Contar jugadores confirmados en cada mitad del campo"""
        if self.field_center_x is None:
            return None, None
        
        left_count = 0
        right_count = 0
        
        for t in self.tracks.values():
            if t['hits'] >= self.min_hits:
                pos_x = t['position'][0]
                if pos_x < self.field_center_x:
                    left_count += 1
                else:
                    right_count += 1
        
        return left_count, right_count
    
    def _can_create_in_side(self, position_x):
        """Verificar si se puede crear un nuevo track en esta mitad del campo"""
        if self.field_center_x is None:
            return True
        
        left_count, right_count = self._count_players_per_side()
        
        if position_x < self.field_center_x:
            return left_count < 2
        else:
            return right_count < 2
    
    def _would_cross_field(self, track, new_position_x):
        """Verificar si un match haría que el jugador cruce al otro lado del campo"""
        if self.field_center_x is None:
            return False
        
        # Aplicar incluso para tracks no confirmados si tienen al menos 1 hit
        if track['hits'] < 1:
            return False
        
        # Zona de tolerancia alrededor del centro (±100px)
        # Los jugadores pueden rondar el centro sin que se considere cruce
        tolerance = 100
        left_boundary = self.field_center_x - tolerance
        right_boundary = self.field_center_x + tolerance
        
        old_x = track['position'][0]
        
        # Determinar en qué zona CLARA está el jugador
        # Zona izquierda clara: < left_boundary
        # Zona neutral: left_boundary <= x < right_boundary
        # Zona derecha clara: >= right_boundary
        
        old_in_left_zone = old_x < left_boundary
        old_in_right_zone = old_x >= right_boundary
        
        new_in_left_zone = new_position_x < left_boundary
        new_in_right_zone = new_position_x >= right_boundary
        
        # Solo es cruce si pasa de una zona CLARA a otra zona CLARA opuesta
        crosses_left_to_right = old_in_left_zone and new_in_right_zone
        crosses_right_to_left = old_in_right_zone and new_in_left_zone
        
        return crosses_left_to_right or crosses_right_to_left
    
    def update(self, frame, detections):
        self.frame_count += 1
        self._init_frame(frame)
        
        for track in self.tracks.values():
            track['predicted_pos'] = self._predict_position(track)
            track['matched'] = False
        
        det_feats = [{'bbox': (d[0],d[1],d[2],d[3]), 'position': (d[4],d[5]),
                      'histogram': self._extract_histogram(frame, (d[0],d[1],d[2],d[3]))} 
                     for d in detections]
        
        if not self.tracks:
            # Crear tracks iniciales: ordenar por X y tomar los correctos de cada lado
            if self.field_center_x and len(det_feats) >= self.max_players:
                det_feats_sorted = sorted(det_feats, key=lambda df: df['position'][0])
                left_dets = [df for df in det_feats_sorted if df['position'][0] < self.field_center_x][:2]
                right_dets = [df for df in det_feats_sorted if df['position'][0] >= self.field_center_x][:2]
                for df in left_dets + right_dets:
                    self._create_track(df)
            else:
                for df in det_feats[:self.max_players]:
                    self._create_track(df)
            return self._get_active()
        
        if not det_feats:
            self._age_tracks()
            return self._get_active()
        
        track_ids = list(self.tracks.keys())
        cost = np.full((len(track_ids), len(det_feats)), 1000.0)
        
        for i, tid in enumerate(track_ids):
            t = self.tracks[tid]
            for j, df in enumerate(det_feats):
                # FILTRO PREVIO: No calcular similaridad si cruza campo (aplicar a TODOS los tracks con hits >= 1)
                if t['hits'] >= 1 and self._would_cross_field(t, df['position'][0]):
                    cost[i, j] = 1000.0  # Costo prohibitivo
                    continue
                
                sim, pos_s, _ = self._calc_similarity(t, df, t['predicted_pos'])
                if pos_s > 0.1:
                    cost[i, j] = 1 - sim
        
        row_idx, col_idx = linear_sum_assignment(cost)
        matched_t, matched_d = set(), set()
        
        # UMBRAL MÁS ESTRICTO: 0.5 en lugar de 0.7
        for i, j in zip(row_idx, col_idx):
            if cost[i, j] < 0.5:
                # Validación adicional: distancia absoluta
                t = self.tracks[track_ids[i]]
                df = det_feats[j]
                dist = np.sqrt((df['position'][0] - t['positions'][-1][0])**2 + 
                               (df['position'][1] - t['positions'][-1][1])**2)
                frames_missed = max(1, t['age'] + 1)
                max_allowed = self.diagonal * self.max_movement * frames_missed * 2
                
                # VALIDACIÓN CRÍTICA: impedir cruces de campo
                if self._would_cross_field(t, df['position'][0]):
                    continue
                
                if dist <= min(250, max_allowed):
                    self._update_track(t, df)
                    matched_t.add(i)
                    matched_d.add(j)
        
        unmatched_d = [j for j in range(len(det_feats)) if j not in matched_d]
        unmatched_t = [i for i in range(len(track_ids)) if i not in matched_t]
        confirmed = len([t for t in self.tracks.values() if t['hits'] >= self.min_hits])
        
        for j in unmatched_d:
            det_x = det_feats[j]['position'][0]
            
            # Verificar si podemos crear un nuevo track en esta mitad del campo
            if confirmed < self.max_players and len(self.tracks) < self.max_players:
                if self._can_create_in_side(det_x):
                    self._create_track(det_feats[j])
                    confirmed += 1
            else:
                # Reasignación más conservadora: solo si el costo es muy bajo
                best_i, best_c = None, 0.6
                for i in unmatched_t:
                    if cost[i, j] < best_c:
                        t = self.tracks[track_ids[i]]
                        df = det_feats[j]
                        
                        # VALIDACIÓN: No permitir reasignación si cruza campo
                        if self._would_cross_field(t, df['position'][0]):
                            continue
                        
                        # Validar distancia antes de reasignar
                        dist = np.sqrt((df['position'][0] - t['positions'][-1][0])**2 + 
                                       (df['position'][1] - t['positions'][-1][1])**2)
                        if dist < 200:  # Máximo 200px para reasignaciones
                            best_c, best_i = cost[i, j], i
                
                if best_i is not None:
                    self._update_track(self.tracks[track_ids[best_i]], det_feats[j])
                    unmatched_t.remove(best_i)
        
        self._age_tracks(det_feats)
        return self._get_active()
    
    def _age_tracks(self, det_feats=None):
        to_del = []
        for tid, t in self.tracks.items():
            if not t['matched']:
                t['age'] += 1
                
                # Si es un track confirmado y hay detecciones del lado opuesto sin asignar,
                # reducir max_age para liberar el slot más rápido
                base_max_age = self.max_age * 3 if t['hits'] >= self.min_hits else self.max_age
                max_age = base_max_age
                
                if self.field_center_x is not None and t['hits'] >= self.min_hits and det_feats:
                    # Determinar lado del track
                    track_x = t['position'][0]
                    tolerance = 100
                    track_in_left = track_x < (self.field_center_x - tolerance)
                    track_in_right = track_x >= (self.field_center_x + tolerance)
                    
                    # Ver si hay detecciones sin asignar del lado opuesto
                    if track_in_left:
                        # Track en izquierda, ¿hay detecciones en derecha?
                        opposite_dets = [df for df in det_feats if df['position'][0] >= (self.field_center_x + tolerance)]
                        if opposite_dets:
                            max_age = self.max_age  # Reducir a 15 frames
                    elif track_in_right:
                        # Track en derecha, ¿hay detecciones en izquierda?
                        opposite_dets = [df for df in det_feats if df['position'][0] < (self.field_center_x - tolerance)]
                        if opposite_dets:
                            max_age = self.max_age  # Reducir a 15 frames
                
                if t['age'] > max_age:
                    to_del.append(tid)
        for tid in to_del:
            del self.tracks[tid]
    
    def _get_active(self):
        return [(t['id'], *t['bbox'], *t['position']) 
                for t in self.tracks.values() if t['hits'] >= self.min_hits]

print("✓ PlayerTracker con límite de 2 jugadores por mitad")

✓ PlayerTracker con límite de 2 jugadores por mitad


# Selección de puntos del campo

In [168]:
video = cv2.VideoCapture(VIDEO_PATH)
if not video.isOpened():
    raise RuntimeError(f"No se pudo abrir el video: {VIDEO_PATH}")

fps = video.get(cv2.CAP_PROP_FPS)
total_frames = int(video.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(video.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(video.get(cv2.CAP_PROP_FRAME_HEIGHT))
print(f"✓ Video: {width}x{height}, {fps:.1f} FPS, {total_frames} frames")

ret, first_frame = video.read()
if not ret:
    raise RuntimeError("No se pudo leer el primer frame")
video.release()

# Selección de puntos del campo en el video
puntos_campo = []
imgA = first_frame.copy()
cv2.namedWindow("Selecciona 4 esquinas del campo", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Selecciona 4 esquinas del campo", 1200, 800)
cv2.imshow("Selecciona 4 esquinas del campo", imgA)
cv2.setMouseCallback("Selecciona 4 esquinas del campo", get_points,
    {"points": puntos_campo, "image": imgA, "wname": "Selecciona 4 esquinas del campo", "max_points": 4})
print("\n📍 Marca las 4 esquinas del CAMPO (arriba-izq → arriba-der → abajo-der → abajo-izq)")
cv2.waitKey(0)
cv2.destroyAllWindows()
puntos_campo = np.array(puntos_campo, dtype=np.float32)
print(f"✓ {len(puntos_campo)} puntos del campo seleccionados")

# Zona de detección expandida
puntos_arena = expand_field_zone(puntos_campo, DETECTION_ZONE_EXPAND_X, DETECTION_ZONE_EXPAND_Y)

# Visualizar
vis_frame = first_frame.copy()
cv2.polylines(vis_frame, [puntos_campo.astype(int)], True, (0, 255, 0), 3)
cv2.polylines(vis_frame, [puntos_arena.astype(int)], True, (0, 255, 255), 3)
cv2.putText(vis_frame, "VERDE = Campo | AMARILLO = Zona deteccion", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
cv2.namedWindow("Zona de deteccion", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Zona de deteccion", 1200, 800)
cv2.imshow("Zona de deteccion", vis_frame)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Guardar puntos del campo
field_data = {'field_points': puntos_campo.tolist(), 'detection_zone': puntos_arena.tolist()}
with open(f"{OUTPUT_FIELD_DIR}/field_points.json", 'w') as f:
    json.dump(field_data, f, indent=2)
print(f"✓ Guardado en {OUTPUT_FIELD_DIR}/field_points.json")

field_center_x = int(np.mean(puntos_campo[:, 0]))

✓ Video: 1920x1080, 30.0 FPS, 441 frames

📍 Marca las 4 esquinas del CAMPO (arriba-izq → arriba-der → abajo-der → abajo-izq)
✓ 4 puntos del campo seleccionados
✓ Guardado en outputs/field/field_points.json


# Selección de puntos del Mapa

In [169]:
# Selección de puntos del mapa (solo si no existe map_points.json)
mapa = cv2.imread(MAPA_PATH)
if mapa is None:
    raise FileNotFoundError(f"No se pudo cargar el mapa: {MAPA_PATH}")

if os.path.exists(MAP_POINTS_JSON):
    with open(MAP_POINTS_JSON, 'r') as f:
        map_data = json.load(f)
    puntos_mapa = np.array(map_data['map_points'], dtype=np.float32)
    print(f"✓ Puntos del mapa cargados desde {MAP_POINTS_JSON}")
else:
    puntos_mapa = []
    imgB = mapa.copy()
    cv2.namedWindow("Selecciona 4 puntos en el mapa", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("Selecciona 4 puntos en el mapa", 1200, 800)
    cv2.imshow("Selecciona 4 puntos en el mapa", imgB)
    cv2.setMouseCallback("Selecciona 4 puntos en el mapa", get_points,
        {"points": puntos_mapa, "image": imgB, "wname": "Selecciona 4 puntos en el mapa", "max_points": 4})
    print("\n📍 Marca los mismos 4 puntos en el MAPA (mismo orden)")
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    puntos_mapa = np.array(puntos_mapa, dtype=np.float32)
    
    with open(MAP_POINTS_JSON, 'w') as f:
        json.dump({'map_points': puntos_mapa.tolist()}, f, indent=2)
    print(f"✓ Guardado en {MAP_POINTS_JSON}")

# Calcular homografía
H, _ = cv2.findHomography(puntos_campo, puntos_mapa)
print(f"✓ Homografía calculada")

✓ Puntos del mapa cargados desde map_points.json
✓ Homografía calculada


# Detección de jugadores

In [176]:
tracker = PlayerTracker(
    max_age=TRACKER_MAX_AGE,
    min_hits=TRACKER_MIN_HITS,
    iou_threshold=TRACKER_IOU_THRESHOLD,
    color_weight=TRACKER_COLOR_WEIGHT,
    position_weight=TRACKER_POSITION_WEIGHT,
    max_players=EXPECTED_PLAYERS,
    max_movement_percent=TRACKER_MAX_MOVEMENT,
    field_center_x=field_center_x
)

video = cv2.VideoCapture(VIDEO_PATH)
video.set(cv2.CAP_PROP_POS_FRAMES, 0)
tracking_data = {}

print(f"\nProcesando {total_frames} frames...")
print(f"  • Máx jugadores: {EXPECTED_PLAYERS} (2 por mitad)")
print(f"  • Centro del campo: X={field_center_x}")
print(f"  • Peso posición: {TRACKER_POSITION_WEIGHT*100:.0f}%, peso color: {TRACKER_COLOR_WEIGHT*100:.0f}%")
print(f"  • Movimiento máximo/frame: {TRACKER_MAX_MOVEMENT*100:.0f}% diagonal\n")

frame_idx = 0

while True:
    ret, frame = video.read()
    if not ret:
        break
    
    # YOLO detecta personas
    results = model(frame, verbose=False, classes=[0])
    
    # Extraer detecciones
    yolo_detections = []
    for r in results:
        if r.boxes is None or len(r.boxes) == 0:
            continue
            
        for box in r.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cx = (x1 + x2) // 2
            cy = y2
            
            # Verificar zona de detección
            in_arena = cv2.pointPolygonTest(puntos_arena.astype(np.int32), (cx, cy), False) >= 0
            
            if in_arena:
                yolo_detections.append((x1, y1, x2, y2, cx, cy))
    
    # PlayerTracker asigna IDs (posición prioritaria)
    tracked_players = tracker.update(frame, yolo_detections)
    tracking_data[frame_idx] = tracked_players
    
    if frame_idx % 100 == 0:
        progress = (frame_idx / total_frames) * 100
        n_tracks = len(tracker.tracks)
        n_confirmed = len([t for t in tracker.tracks.values() if t['hits'] >= TRACKER_MIN_HITS])
        print(f"  Frame {frame_idx}/{total_frames} ({progress:.0f}%) - {len(tracked_players)} jugadores | Tracks: {n_tracks} (confirmados: {n_confirmed})")
    
    frame_idx += 1

video.release()

print(f"\n✓ Tracking completado")

# Estadísticas
all_ids = set()
for dets in tracking_data.values():
    for det in dets:
        all_ids.add(det[0])

final_ids = sorted(all_ids)
total_detections_robust = sum(len(dets) for dets in tracking_data.values())

print(f"  IDs detectados: {final_ids} ({len(final_ids)} únicos)")
print(f"  Total detecciones: {total_detections_robust}")

# Verificar si se respetó el límite
if len(final_ids) <= EXPECTED_PLAYERS:
    print(f"  ✅ Límite de {EXPECTED_PLAYERS} jugadores respetado")
else:
    print(f"  ⚠️  Se detectaron más IDs de lo esperado ({len(final_ids)} vs {EXPECTED_PLAYERS})")

# Análisis de frames por ID
id_frame_counts = defaultdict(int)
for dets in tracking_data.values():
    for det in dets:
        id_frame_counts[det[0]] += 1

print(f"\n  Frames por jugador:")
for track_id in sorted(id_frame_counts.keys()):
    frames = id_frame_counts[track_id]
    percentage = (frames / total_frames) * 100
    status = "✓" if percentage > 50 else "⚠️"
    print(f"    {status} ID {track_id}: {frames} frames ({percentage:.0f}%)")

print()


Procesando 441 frames...
  • Máx jugadores: 4 (2 por mitad)
  • Centro del campo: X=984
  • Peso posición: 80%, peso color: 20%
  • Movimiento máximo/frame: 5% diagonal

  Frame 0/441 (0%) - 0 jugadores | Tracks: 3 (confirmados: 0)
  Frame 100/441 (23%) - 4 jugadores | Tracks: 4 (confirmados: 4)
  Frame 200/441 (45%) - 4 jugadores | Tracks: 4 (confirmados: 4)
  Frame 300/441 (68%) - 4 jugadores | Tracks: 4 (confirmados: 4)
  Frame 400/441 (91%) - 4 jugadores | Tracks: 4 (confirmados: 4)

✓ Tracking completado
  IDs detectados: [1, 2, 3, 4, 5] (5 únicos)
  Total detecciones: 1751
  ⚠️  Se detectaron más IDs de lo esperado (5 vs 4)

  Frames por jugador:
    ⚠️ ID 1: 35 frames (8%)
    ✓ ID 2: 439 frames (100%)
    ✓ ID 3: 438 frames (99%)
    ✓ ID 4: 438 frames (99%)
    ✓ ID 5: 401 frames (91%)



# Generación del vídeo resultado

In [171]:
# Cargar video y mapa
video = cv2.VideoCapture(VIDEO_PATH)
mapa = cv2.imread(MAPA_PATH)

# Configuración del video de salida
output_filename = f"{OUTPUT_TRACKING_DIR}/player_tracking.mp4"
fourcc = cv2.VideoWriter_fourcc(*'mp4v')

# Dimensiones: video arriba, mapa abajo
map_display_width = width // 2
map_display_height = int(mapa.shape[0] * (map_display_width / mapa.shape[1]))
output_height = height + map_display_height
output_width = width

out = cv2.VideoWriter(output_filename, fourcc, fps, (output_width, output_height))

frame_idx = 0
video.set(cv2.CAP_PROP_POS_FRAMES, 0)

trajectories = defaultdict(list)

while True:
    ret, frame = video.read()
    if not ret:
        break
    
    # VIDEO CON TRACKING
    tracking_frame = frame.copy()
    
    # Dibujar zonas
    cv2.polylines(tracking_frame, [puntos_arena.astype(np.int32)], True, (0, 255, 255), 2)
    cv2.polylines(tracking_frame, [puntos_campo.astype(np.int32)], True, (0, 255, 0), 2)
    
    # Dibujar jugadores
    if frame_idx in tracking_data:
        detections = tracking_data[frame_idx]
        
        for det in detections:
            track_id, x1, y1, x2, y2, cx, cy = det
            
            color = get_player_color(track_id, final_ids, cx, field_center_x)
            
            cv2.rectangle(tracking_frame, (x1, y1), (x2, y2), color, 2)
            cv2.circle(tracking_frame, (cx, cy), 5, color, -1)
            
            label = f"ID {track_id}"
            cv2.putText(tracking_frame, label, (x1, y1 - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
            
            # Guardar trayectoria
            point_video = np.array([[[cx, cy]]], dtype=np.float32)
            point_mapa = cv2.perspectiveTransform(point_video, H)
            field_x, field_y = point_mapa[0][0]
            trajectories[track_id].append((int(field_x), int(field_y), cx))
    
    # Info del frame
    num_players = len(tracking_data.get(frame_idx, []))
    info_text = f"Frame: {frame_idx}/{total_frames} | Jugadores: {num_players}"
    cv2.putText(tracking_frame, info_text, (10, 30),
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    # MAPA CON POSICIONES
    mapa_resized = cv2.resize(mapa, (map_display_width, map_display_height))
    mapa_display = mapa_resized.copy()
    
    if frame_idx in tracking_data:
        detections = tracking_data[frame_idx]
        
        for det in detections:
            track_id, x1, y1, x2, y2, cx, cy = det
            
            color = get_player_color(track_id, final_ids, cx, field_center_x)
            
            point_video = np.array([[[cx, cy]]], dtype=np.float32)
            point_mapa_pos = cv2.perspectiveTransform(point_video, H)
            field_x, field_y = point_mapa_pos[0][0]
            
            scale_x = map_display_width / mapa.shape[1]
            scale_y = map_display_height / mapa.shape[0]
            
            current_pos = (int(field_x * scale_x), int(field_y * scale_y))
            
            cv2.circle(mapa_display, current_pos, 8, color, -1)
            cv2.circle(mapa_display, current_pos, 10, (255, 255, 255), 2)
            cv2.putText(mapa_display, f"{track_id}", 
                       (current_pos[0] + 12, current_pos[1] - 10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)
    
    # COMBINAR
    combined_frame = np.zeros((output_height, output_width, 3), dtype=np.uint8)
    combined_frame[0:height, 0:width] = tracking_frame
    
    map_x_offset = (output_width - map_display_width) // 2
    combined_frame[height:height+map_display_height, 
                   map_x_offset:map_x_offset+map_display_width] = mapa_display
    
    out.write(combined_frame)
    
    if frame_idx % 100 == 0:
        progress = (frame_idx / total_frames) * 100
    
    frame_idx += 1

video.release()
out.release()

# Exportación a CSV

In [172]:
tracking_list = []

for frame_idx, detections in sorted(tracking_data.items()):
    for det in detections:
        track_id, x1, y1, x2, y2, cx, cy = det
        
        point_video = np.array([[[cx, cy]]], dtype=np.float32)
        point_mapa = cv2.perspectiveTransform(point_video, H)
        field_x, field_y = point_mapa[0][0]
        
        tracking_list.append({
            'frame': frame_idx,
            'track_id': track_id,
            'bbox_x1': x1,
            'bbox_y1': y1,
            'bbox_x2': x2,
            'bbox_y2': y2,
            'center_x': cx,
            'center_y': cy,
            'field_x': float(field_x),
            'field_y': float(field_y),
            'timestamp_sec': frame_idx / fps
        })

# CSV
csv_filename = f"{OUTPUT_TRACKING_DIR}/tracking_data.csv"
with open(csv_filename, 'w', newline='', encoding='utf-8') as csvfile:
    fieldnames = ['frame', 'track_id', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2', 
                  'center_x', 'center_y', 'field_x', 'field_y', 'timestamp_sec']
    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(tracking_list)

# JSON - Detecciones por ID
detections_per_id = {}
for tid in sorted(final_ids):
    detections_per_id[str(tid)] = id_frame_counts[tid]

summary_data = {
    'video_info': {
        'fps': fps,
        'total_frames': total_frames,
        'width': width,
        'height': height
    },
    'homography_matrix': H.tolist(),
    'player_ids': sorted(final_ids),
    'num_players': len(final_ids),
    'total_detections': {
        'total': len(tracking_list),
        'per_player': detections_per_id
    },
    'tracker_config': {
        'max_age': TRACKER_MAX_AGE,
        'min_hits': TRACKER_MIN_HITS,
        'min_similarity': TRACKER_MIN_SIMILARITY,
        'color_weight': TRACKER_COLOR_WEIGHT,
        'position_weight': TRACKER_POSITION_WEIGHT
    }
}

json_filename = f"{OUTPUT_TRACKING_DIR}/tracking_summary.json"
with open(json_filename, 'w', encoding='utf-8') as jsonfile:
    json.dump(summary_data, jsonfile, indent=2)

# Generación de trayectorias

In [173]:
mapa_traj = cv2.imread(MAPA_PATH)

# Reconstruir trayectorias
full_trajectories = defaultdict(list)

for frame_idx in sorted(tracking_data.keys()):
    for det in tracking_data[frame_idx]:
        track_id, x1, y1, x2, y2, cx, cy = det
        
        point_video = np.array([[[cx, cy]]], dtype=np.float32)
        point_mapa = cv2.perspectiveTransform(point_video, H)
        field_x, field_y = point_mapa[0][0]
        
        full_trajectories[track_id].append((int(field_x), int(field_y), cx))

# Dibujar trayectorias combinadas
for track_id in sorted(full_trajectories.keys()):
    trajectory = full_trajectories[track_id]
    
    if len(trajectory) < 2:
        continue
    
    # Líneas
    for i in range(1, len(trajectory)):
        field_x1, field_y1, cx1 = trajectory[i-1]
        field_x2, field_y2, cx2 = trajectory[i]
        
        avg_cx = (cx1 + cx2) / 2
        color = get_player_color(track_id, final_ids, avg_cx, field_center_x)
        
        cv2.line(mapa_traj, (field_x1, field_y1), (field_x2, field_y2), color, 3)
    
    # Punto inicio
    start_x, start_y, start_cx = trajectory[0]
    start_color = get_player_color(track_id, final_ids, start_cx, field_center_x)
    
    cv2.circle(mapa_traj, (start_x, start_y), 12, (255, 255, 255), -1)
    cv2.circle(mapa_traj, (start_x, start_y), 8, start_color, -1)
    cv2.putText(mapa_traj, f"{track_id}", (start_x - 10, start_y - 15),
               cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    
    # Punto final
    end_x, end_y, end_cx = trajectory[-1]
    end_color = get_player_color(track_id, final_ids, end_cx, field_center_x)
    
    cv2.rectangle(mapa_traj, (end_x-8, end_y-8), (end_x+8, end_y+8), end_color, -1)
    cv2.rectangle(mapa_traj, (end_x-10, end_y-10), (end_x+10, end_y+10), (255, 255, 255), 2)

# Leyenda con colores diferenciados
cv2.putText(mapa_traj, "EQUIPO IZQUIERDO", (20, 30),
           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 100, 0), 2)
cv2.circle(mapa_traj, (350, 25), 8, (255, 100, 0), -1)  # Azul claro
cv2.circle(mapa_traj, (390, 25), 8, (180, 0, 0), -1)    # Azul oscuro

cv2.putText(mapa_traj, "EQUIPO DERECHO", (20, 65),
           cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 100, 255), 2)
cv2.circle(mapa_traj, (330, 60), 8, (0, 100, 255), -1)  # Rojo claro
cv2.circle(mapa_traj, (370, 60), 8, (0, 0, 180), -1)    # Rojo oscuro

# Guardar trayectorias combinadas
traj_filename = f"{OUTPUT_TRACKING_DIR}/trayectorias.png"
cv2.imwrite(traj_filename, mapa_traj)

# Generar PNGs individuales por jugador
if GENERATE_INDIVIDUAL_TRAJECTORIES:
    
    for track_id in sorted(full_trajectories.keys()):
        trajectory = full_trajectories[track_id]
        
        if len(trajectory) < 2:
            continue
        
        # Crear mapa limpio para este jugador
        mapa_individual = cv2.imread(MAPA_PATH)
        
        # Determinar color del jugador
        avg_cx = np.mean([t[2] for t in trajectory])
        color = get_player_color(track_id, final_ids, avg_cx, field_center_x)
        
        # Dibujar trayectoria
        for i in range(1, len(trajectory)):
            field_x1, field_y1, _ = trajectory[i-1]
            field_x2, field_y2, _ = trajectory[i]
            cv2.line(mapa_individual, (field_x1, field_y1), (field_x2, field_y2), color, 3)
        
        # Punto inicio (círculo)
        start_x, start_y, _ = trajectory[0]
        cv2.circle(mapa_individual, (start_x, start_y), 12, (255, 255, 255), -1)
        cv2.circle(mapa_individual, (start_x, start_y), 8, color, -1)
        cv2.putText(mapa_individual, "INICIO", (start_x - 25, start_y - 18),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        
        # Punto final (cuadrado)
        end_x, end_y, _ = trajectory[-1]
        cv2.rectangle(mapa_individual, (end_x-8, end_y-8), (end_x+8, end_y+8), color, -1)
        cv2.rectangle(mapa_individual, (end_x-10, end_y-10), (end_x+10, end_y+10), (255, 255, 255), 2)
        cv2.putText(mapa_individual, "FIN", (end_x - 12, end_y - 18),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
        
        # Título con ID del jugador
        equipo = "IZQUIERDO" if avg_cx < field_center_x else "DERECHO"
        cv2.putText(mapa_individual, f"JUGADOR {track_id} - EQUIPO {equipo}", (20, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, color, 2)
        
        # Guardar
        individual_filename = f"{OUTPUT_TRACKING_DIR}/trayectoria_jugador_{track_id}.png"
        cv2.imwrite(individual_filename, mapa_individual)
        print(f"  ✓ {individual_filename}")

print()

  ✓ outputs/player_tracking/trayectoria_jugador_1.png
  ✓ outputs/player_tracking/trayectoria_jugador_2.png
  ✓ outputs/player_tracking/trayectoria_jugador_3.png
  ✓ outputs/player_tracking/trayectoria_jugador_4.png
  ✓ outputs/player_tracking/trayectoria_jugador_5.png



# Resumen Final y presentación en vídeo

In [174]:
# Estadísticas
frames_with_n_players = sum(1 for dets in tracking_data.values() if len(dets) == EXPECTED_PLAYERS)
quality_percent = (frames_with_n_players / len(tracking_data)) * 100 if tracking_data else 0

print(f"\n📊 Resumen:")
print(f"   • {total_frames} frames ({total_frames/fps:.1f}s)")
print(f"   • {len(final_ids)} jugadores: {final_ids}")
print(f"   • {frames_with_n_players}/{len(tracking_data)} frames perfectos ({quality_percent:.1f}%)")
print(f"   • {total_detections_robust} detecciones")

print(f"\n📁 Archivos:")
print(f"   • {OUTPUT_FIELD_DIR}/field_points.json")
print(f"   • {MAP_POINTS_JSON}")
print(f"   • {OUTPUT_TRACKING_DIR}/tracking_data.csv")
print(f"   • {OUTPUT_TRACKING_DIR}/tracking_summary.json")
print(f"   • {OUTPUT_TRACKING_DIR}/player_tracking.mp4")
print(f"   • {OUTPUT_TRACKING_DIR}/trayectorias.png")

# Reproducción con cv2
if AUTO_OPEN_VIDEO:
    print(f"\n" + "=" * 70)
    print("🎬 REPRODUCIENDO VIDEO")
    print("=" * 70)
    video_path = f"{OUTPUT_TRACKING_DIR}/player_tracking.mp4"
    
    if os.path.exists(video_path):
        video_playback = cv2.VideoCapture(video_path)
        
        if not video_playback.isOpened():
            print(f"\n⚠️  Error al abrir el video")
        else:
            print(f"\nControles: ESC=Salir | ESPACIO=Pausa | R=Reiniciar\n")
            
            cv2.namedWindow("Tracking", cv2.WINDOW_NORMAL)
            cv2.resizeWindow("Tracking", 1200, 900)
            
            paused = False
            
            while True:
                if not paused:
                    ret, frame = video_playback.read()
                    if not ret:
                        video_playback.set(cv2.CAP_PROP_POS_FRAMES, 0)
                        continue
                    
                    cv2.imshow("Tracking", frame)
                
                key = cv2.waitKey(30) & 0xFF
                
                if key == 27:  # ESC
                    break
                elif key == 32:  # ESPACIO
                    paused = not paused
                elif key == ord('r') or key == ord('R'):
                    video_playback.set(cv2.CAP_PROP_POS_FRAMES, 0)
                    paused = False
            
            video_playback.release()
            cv2.destroyAllWindows()
            
            print("\n✓ Visualización completada")
    else:
        print(f"\n⚠️  Video no encontrado: {video_path}")
else:
    print(f"\n💡 Cambia AUTO_OPEN_VIDEO = True para visualizar automáticamente")

print()


📊 Resumen:
   • 441 frames (14.7s)
   • 5 jugadores: [1, 2, 3, 4, 5]
   • 435/441 frames perfectos (98.6%)
   • 1751 detecciones

📁 Archivos:
   • outputs/field/field_points.json
   • map_points.json
   • outputs/player_tracking/tracking_data.csv
   • outputs/player_tracking/tracking_summary.json
   • outputs/player_tracking/player_tracking.mp4
   • outputs/player_tracking/trayectorias.png

🎬 REPRODUCIENDO VIDEO

Controles: ESC=Salir | ESPACIO=Pausa | R=Reiniciar


✓ Visualización completada

